# Watsonx MLOps Model Monitor - Demo

This notebook demonstrates the core capabilities of the Watsonx MLOps platform:

1. **Model Training** - Train a classifier with scikit-learn pipelines
2. **Drift Detection** - Detect data drift using PSI, KL, JS, and KS tests
3. **Fairness Monitoring** - Audit model fairness across protected groups
4. **Performance Tracking** - Monitor accuracy, F1, precision, recall over time
5. **Model Serving** - Serve models through the gateway with A/B routing
6. **Alert Engine** - Fire alerts on drift and fairness violations

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
print("Environment ready.")

## 1. Model Training Pipeline

Train a binary classifier using the `ModelTrainer` with a scikit-learn pipeline
(StandardScaler + RandomForestClassifier). The trainer automatically computes
accuracy, F1, precision, recall, ROC-AUC, and runs cross-validation.

In [ ]:
from sklearn.datasets import make_classification
from src.training.trainer import ModelTrainer

# Generate synthetic credit risk dataset
X, y = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=2,
    random_state=42,
    weights=[0.7, 0.3],
)

feature_names = [
    "age", "income", "debt_ratio", "credit_score",
    "employment_years", "num_accounts", "loan_amount", "monthly_payment",
]

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

In [ ]:
# Train with Random Forest
trainer = ModelTrainer(
    algorithm="random_forest",
    params={"n_estimators": 100, "max_depth": 10},
    random_state=42,
)
result = trainer.train(X, y, feature_names=feature_names)

print("Training Metrics:")
for metric, value in result.metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nCross-validation F1: {result.cv_scores.mean():.4f} +/- {result.cv_scores.std():.4f}")

## 2. Hyperparameter Optimization

Use `HyperparameterOptimizer` with GridSearchCV to find optimal hyperparameters.

In [ ]:
from src.training.hyperopt import HyperparameterOptimizer

optimizer = HyperparameterOptimizer(
    algorithm="random_forest",
    param_grid={
        "classifier__n_estimators": [50, 100],
        "classifier__max_depth": [5, 10],
        "classifier__min_samples_split": [2, 5],
    },
    cv_folds=3,
)

optimized_result = optimizer.optimize(X, y, feature_names=feature_names)

print("Optimized Parameters:")
for param, value in optimized_result.params.items():
    print(f"  {param}: {value}")

print(f"\nOptimized Accuracy: {optimized_result.metrics['accuracy']:.4f}")
print(f"Optimized F1:       {optimized_result.metrics['f1']:.4f}")

## 3. Drift Detection

Detect data drift between training (reference) and production distributions
using four complementary statistical tests:
- **PSI** (Population Stability Index)
- **KL-divergence** (Kullback-Leibler)
- **JS-distance** (Jensen-Shannon)
- **KS-test** (Kolmogorov-Smirnov)

In [ ]:
from src.monitoring.drift_detector import DriftDetector

detector = DriftDetector(psi_threshold=0.2, n_bins=10)

# Simulate production data with drift in some features
X_production = X.copy()
X_production[:, 0] += np.random.normal(1.5, 0.5, X.shape[0])  # age: significant shift
X_production[:, 1] += np.random.normal(0.05, 0.1, X.shape[0])  # income: minor shift

report = detector.detect(X, X_production, feature_names=feature_names)

print(f"Overall Drift Detected: {report.overall_drifted}")
print(f"Drifted Features: {report.n_drifted_features}/{report.n_total_features}")
print(f"Drift Ratio: {report.drift_ratio:.2%}")
print("\nPer-Feature Results:")
print(f"{'Feature':<20} {'PSI':>8} {'KS-stat':>8} {'KS-pval':>8} {'JS-dist':>8} {'Drifted':>8} {'Severity':>10}")
print("-" * 82)
for r in report.feature_results:
    print(
        f"{r.feature_name:<20} {r.psi:>8.4f} {r.ks_statistic:>8.4f} "
        f"{r.ks_pvalue:>8.4f} {r.js_distance:>8.4f} {str(r.is_drifted):>8} {r.severity.value:>10}"
    )

## 4. Fairness Monitoring

Audit model fairness across protected demographic groups. The `FairnessMonitor`
computes demographic parity, equalized odds, and calibration metrics.

In [ ]:
from src.monitoring.fairness_monitor import FairnessMonitor

monitor = FairnessMonitor(dp_threshold=0.1, eo_threshold=0.1)

# Simulate protected attribute
gender = np.array(["M"] * 500 + ["F"] * 500)
np.random.shuffle(gender)

y_pred = result.model.predict(X)

fairness_report = monitor.evaluate(
    y_true=y,
    y_pred=y_pred,
    protected_attributes={"gender": gender},
)

print(f"Overall Fair: {fairness_report.overall_fair}")
print(f"Total Violations: {fairness_report.n_violations}")

for attr_result in fairness_report.attribute_results:
    print(f"\nAttribute: {attr_result.attribute}")
    print(f"  Demographic Parity Difference: {attr_result.demographic_parity_difference:.4f}")
    print(f"  Demographic Parity Ratio:      {attr_result.demographic_parity_ratio:.4f}")
    print(f"  Equalized Odds Difference:     {attr_result.equalized_odds_difference:.4f}")
    print(f"  Calibration Difference:        {attr_result.calibration_difference:.4f}")
    print(f"  Fair: {attr_result.is_fair}")

    if attr_result.violations:
        print("  Violations:")
        for v in attr_result.violations:
            print(f"    - {v}")

    print("\n  Group Metrics:")
    for gm in attr_result.group_metrics:
        print(
            f"    {gm.group_name}: size={gm.group_size}, "
            f"pos_rate={gm.positive_rate:.3f}, TPR={gm.true_positive_rate:.3f}, "
            f"FPR={gm.false_positive_rate:.3f}"
        )

## 5. Performance Tracking

Track model performance over a sliding window of predictions and detect degradation.

In [ ]:
from src.monitoring.performance_tracker import PerformanceTracker

tracker = PerformanceTracker(
    window_size=200,
    baseline_metrics={"accuracy": 0.90, "f1": 0.85, "precision": 0.85, "recall": 0.80},
)

# Simulate production predictions over time
for batch_idx in range(5):
    batch_size = 50
    start = batch_idx * batch_size
    end = start + batch_size

    y_batch_true = y[start:end].tolist()
    y_batch_pred = y_pred[start:end].tolist()

    tracker.record(y_batch_true, y_batch_pred)

snapshot = tracker.get_current_metrics()
if snapshot:
    print(f"Current Performance (window={snapshot.window_size}, samples={snapshot.n_samples}):")
    print(f"  Accuracy:  {snapshot.accuracy:.4f}")
    print(f"  F1:        {snapshot.f1:.4f}")
    print(f"  Precision: {snapshot.precision:.4f}")
    print(f"  Recall:    {snapshot.recall:.4f}")

# Check for degradation
degradations = tracker.check_degradation()
print("\nDegradation Check:")
for d in degradations:
    status = "DEGRADED" if d.is_degraded else "OK"
    print(f"  {d.metric}: current={d.current_value:.4f}, baseline={d.baseline_value:.4f}, drop={d.drop:.4f} [{status}]")

## 6. Model Serving Gateway

Serve models through the `ModelGateway` and route traffic with the `ABRouter`
using canary, blue-green, or shadow strategies.

In [ ]:
from src.serving.gateway import ModelGateway
from src.serving.ab_router import ABRouter, RoutingConfig, RoutingStrategy

# Set up gateway with production and canary models
gateway = ModelGateway()
gateway.load_model("production", result.model, metadata={"version": 1, "algorithm": "random_forest"})
gateway.load_model("canary", optimized_result.model, metadata={"version": 2, "algorithm": "random_forest_optimized"})

print("Loaded Models:")
for m in gateway.list_models():
    print(f"  {m['model_id']}: {m['metadata']}")

# Direct prediction
sample = X[:3].tolist()
pred = gateway.predict("production", sample)
print(f"\nDirect Prediction: {pred['predictions']}")

In [ ]:
# Canary routing - 20% traffic to candidate
config = RoutingConfig(
    strategy=RoutingStrategy.CANARY,
    canary_weight=0.2,
)
router = ABRouter(gateway, config=config)

# Route 20 requests
routes = []
for i in range(20):
    result_route = router.route([X[i].tolist()])
    routes.append(result_route.routed_to)

stats = router.get_stats()
print(f"Routing Strategy: {stats['strategy']}")
print(f"Canary Weight: {stats['canary_weight']:.0%}")
print(f"Total Requests: {stats['total_requests']}")
print(f"Routing Distribution: {stats['routing_stats']}")

## 7. Shadow Mode

Run a candidate model alongside production without affecting users.
Compare agreement/divergence between models.

In [ ]:
from src.serving.shadow_mode import ShadowRunner

runner = ShadowRunner(gateway)

# Run shadow comparisons
for i in range(100):
    features = [X[i].tolist()]
    prod_pred = gateway.predict("production", features)

    runner.run_shadow(
        "canary",
        features,
        production_predictions=prod_pred["predictions"],
    )

summary = runner.get_summary()
print("Shadow Mode Summary:")
print(f"  Total Requests:   {summary['total_requests']}")
print(f"  Agreement Rate:   {summary['agreement_rate']:.2%}")
print(f"  Divergence Rate:  {summary['divergence_rate']:.2%}")
print(f"  Divergence Count: {summary['divergence_count']}")

## 8. Alert Engine

Fire alerts based on monitoring results with configurable severity and cooldowns.

In [ ]:
from src.monitoring.alert_engine import AlertEngine, AlertSeverity

engine = AlertEngine(cooldown_seconds=1)

# Fire alerts based on monitoring results
if report.overall_drifted:
    for r in report.feature_results:
        if r.is_drifted:
            engine.fire(
                severity=AlertSeverity.WARNING if r.severity.value == "moderate" else AlertSeverity.CRITICAL,
                category="drift",
                title=f"Drift detected: {r.feature_name}",
                message=f"PSI={r.psi:.4f}, JS={r.js_distance:.4f}, severity={r.severity.value}",
                details={"psi": r.psi, "js_distance": r.js_distance},
            )

if not fairness_report.overall_fair:
    for attr_result in fairness_report.attribute_results:
        for violation in attr_result.violations:
            engine.fire(
                severity=AlertSeverity.CRITICAL,
                category="fairness",
                title=f"Fairness violation: {attr_result.attribute}",
                message=violation,
            )

# Show alert summary
summary = engine.get_summary()
print("Alert Engine Summary:")
print(f"  Total Alerts:    {summary['total_alerts']}")
print(f"  Unacknowledged:  {summary['unacknowledged']}")
print(f"  By Severity:     {summary['by_severity']}")
print(f"  By Category:     {summary['by_category']}")

# Show recent alerts
print("\nRecent Alerts:")
for alert in engine.get_history(limit=5):
    print(f"  [{alert.severity.value.upper()}] {alert.title}: {alert.message}")

## Summary

This demo covered the full MLOps lifecycle:

| Stage | Component | Status |
|---|---|---|
| Training | ModelTrainer (Random Forest) | Trained with metrics |
| Optimization | HyperparameterOptimizer (GridSearchCV) | Best params found |
| Drift Detection | DriftDetector (PSI/KL/JS/KS) | Drift detected on shifted features |
| Fairness | FairnessMonitor (DP/EO/Calibration) | Audited across groups |
| Performance | PerformanceTracker (sliding window) | Degradation checked |
| Serving | ModelGateway + ABRouter | Canary routing active |
| Shadow | ShadowRunner | Agreement rate measured |
| Alerts | AlertEngine | Alerts fired on violations |

In production, IBM Watsonx Granite generates human-readable narratives from these
monitoring signals, providing stakeholders with clear explanations of model behavior.